In [2]:
import boto3
import sagemaker

In [3]:
#check role
role = sagemaker.get_execution_role()
print(role)

In [4]:
#check your bucket
s3 = boto3.client("s3")
response = s3.list_buckets()
for bucket in response['Buckets']:
    print(bucket['Name'])

In [5]:
#package up a trained model to deploy it  AWS SageMaker
#because deploying our model to cloud platforms like AWS SageMaker, Google Cloud Vertex AI, or Azure ML,
#their systems are hardcoded to look for a single compressed archive.

import tarfile
import shutil
import os

os.makedirs("model", exist_ok=True)
shutil.copy("../pipeline/artifacts/RandomForest_model.pkl", "model/model_credit.joblib")
shutil.copy("../pipeline/artifacts/label_encoder.pkl", "model/label_encoder.pkl")

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/model_credit.joblib", arcname="model_credit.joblib")
    tar.add("model/label_encoder.pkl", arcname="label_encoder.pkl")

In [6]:
#Upload model to S3, because deployment will take the model form s3

s3 = boto3.client("s3")

bucket_name = sagemaker.Session().default_bucket()  # sagemaker-<region>-<account-id>

s3.upload_file(
    "model.tar.gz",
    bucket_name,
    "model/model.tar.gz"
)

In [7]:
import subprocess
subprocess.run(["pip", "install", "joblib", "scikit-learn", "numpy", "pandas"], check=True)

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


CompletedProcess(args=['pip', 'install', 'joblib', 'scikit-learn', 'numpy', 'pandas'], returncode=0)

In [8]:
#check model can be run or not

from inference import model_fn

model = model_fn("model")

In [9]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel


# ---- EDIT THESE ---------------------------------------------------------
BUCKET = sagemaker.Session().default_bucket()  # sagemaker-<region>-<account-id>
MODEL_S3_KEY = "model/model.tar.gz"
ENDPOINT_NAME = "credit-endpoint"
# -------------------------------------------------------------------------

REGION = "us-east-1"
INSTANCE_TYPE = "ml.m5.large"
FRAMEWORK_VERSION = "1.4-2"


def get_lab_role_arn() -> str:
    iam = boto3.client("iam")
    return iam.get_role(RoleName="LabRole")["Role"]["Arn"]


def main() -> None:
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn = get_lab_role_arn()
    model_s3_uri = f"s3://{BUCKET}/{MODEL_S3_KEY}"

    print(f"Role:      {role_arn}")
    print(f"Model URI: {model_s3_uri}")
    print(f"Endpoint:  {ENDPOINT_NAME}")

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point="inference.py",
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print("\nDeploying endpoint (5-8 minutes)...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print("Testing endpoint with sample data...")

    sample = {
        "instances": [[1, 40, 120000.0, 10000.0, 3, 3, 5, 1, 2, 1, 5.0, 1, 200.0, 28.0, 300, 50.0, 500.0, 800.0, 1, "Good", "Engineer", "No", "High_spent_Large_value_payments"]]}

    runtime = boto3.client("sagemaker-runtime", region_name=REGION)
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=str(sample).replace("'", '"'),
    )
    print("\nSmoke test response:")
    print(response["Body"].read().decode("utf-8"))


if __name__ == "__main__":
    main()

Role:      arn:aws:iam::<account-id>:role/LabRole
Model URI: s3://sagemaker-us-east-1-<account-id>/model/model.tar.gz
Endpoint:  credit-endpoint

Deploying endpoint (5-8 minutes)...
------!Testing endpoint with sample data...

Smoke test response:
{"probabilities": [[0.6348719274838827, 0.0938138888684256, 0.27131418364769155]], "predictions": [0], "labels": ["Good"]}


In [ ]:
#check log here: https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logsV2:log-groups

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name="us-east-1")
ENDPOINT_NAME = "credit-endpoint"

# 1. Delete the stuck endpoint
print(f"Deleting failed endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deletion triggered.")
except Exception as e:
    print(f"No endpoint found to delete: {e}")

# 2. Delete the conflicting endpoint configuration
print(f"Deleting endpoint configuration: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Endpoint configuration deletion triggered.")
except Exception as e:
    print(f"No config found to delete: {e}")

print("\nCleanup complete! You can now safely run your main deploy script.")

In [ ]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

# 1. Delete all endpoints
print("=== Endpoints ===")
for ep in sm.list_endpoints()['Endpoints']:
    sm.delete_endpoint(EndpointName=ep['EndpointName'])
    print(f"Deleted endpoint: {ep['EndpointName']}")

# 2. Delete all endpoint configs
print("=== Endpoint Configs ===")
for ec in sm.list_endpoint_configs()['EndpointConfigs']:
    sm.delete_endpoint_config(EndpointConfigName=ec['EndpointConfigName'])
    print(f"Deleted config: {ec['EndpointConfigName']}")

# 3. Delete all models
print("=== Models ===")
for m in sm.list_models()['Models']:
    sm.delete_model(ModelName=m['ModelName'])
    print(f"Deleted model: {m['ModelName']}")

print("\nAll SageMaker resources cleared!")
print("Don't forget to manually check S3 buckets and EBS volumes in the console.")

In [ ]:
import boto3
sm = boto3.client("sagemaker", region_name="us-east-1")

print("ENDPOINTS:", sm.list_endpoints()['Endpoints'])
print("CONFIGS:", sm.list_endpoint_configs()['EndpointConfigs'])
print("MODELS:", sm.list_models()['Models'])